# ErisML Compiler — Quickstart

This notebook walks through compiling each of the three example texts end-to-end
through the Phase 1 MVP pipeline.

Pipeline: text → segmentation → MockExtractor → EM-DAG → MoralVector timeline → DEME stub → audit.

For an honest, deterministic, offline run, we use the **MockExtractor** here. The
**RuleExtractor** (Tier 2) works on arbitrary text but with the limits of pattern-based
extraction; the **LLMExtractor** (Tier 3) is a Phase-2 skeleton.


In [1]:
import sys
from pathlib import Path

from erisml_compiler.pipeline.orchestrator import CompileOptions, compile_document
from erisml_compiler.tiers import CompilerTier
from erisml_compiler.streaming.captioner import TerminalCaptioner
from erisml_compiler.streaming.streamer import MoralStreamer

EXAMPLES = Path("..") / "examples" if Path("..").joinpath("examples").exists() else Path("examples")
list(EXAMPLES.glob("*.txt"))

[WindowsPath('../examples/medical_confidentiality.txt'),
 WindowsPath('../examples/nazi_attic.txt'),
 WindowsPath('../examples/whistleblower.txt')]

## 1. Nazi attic (canonical example from spec §28)

In [2]:
ir_nazi = compile_document(
    EXAMPLES / "nazi_attic.txt",
    CompileOptions(tier=CompilerTier.RULES, extractor="mock"),
)
print(f"Verdict: {ir_nazi.deme_verdict.verdict}")
print(f"Confidence: {ir_nazi.deme_verdict.confidence}")
print(f"Canonical form: {ir_nazi.canonical_form}")
print(f"IR hash: {ir_nazi.audit.ir_hash}")

Verdict: tragic_conflict_escalate
Confidence: 0.85
Canonical form: coercive_murderous_interrogation_with_collective_reprisal
IR hash: 72d052bcc0c4b00ab6f398faa9d1bef2e1072aa4a7289f0afbca7e4f074311f6


In [3]:
# Stream the compiled IR as real-time captions.
TerminalCaptioner().render(MoralStreamer(ir_nazi))

[#] Document loaded: nazi_attic
[t=0] vow_made  actor=speaker  target=hidden_refugees  content='conceal hiding place'
[t=1] threat_uttered  actor=nazis  target=village  content='murder entire village if lied to'
[t=2] demand_made  actor=nazis  target=speaker  content='reveal location of hidden refugees'
  >  fact fact_001: legitimacy: Authority is coercive and tyrannical; legitimacy void.
  >  fact fact_002: coercion: Soldiers threaten murderous reprisal against village.
  >  fact fact_003: externality: Catastrophic non-consensual risk imposed on village.
  >  fact fact_004: care: Speaker is actively protecting vulnerable hidden refugees.
  >  fact fact_005: deception: Deception toward illegitimate murderous authority is permitted.
  *  EM[autonomy] value=-1.000 conf=0.95
  *  EM[care] value=+0.850 conf=0.93
  *  EM[epistemic] value=-0.500 conf=0.90
  *  EM[externality] value=-1.000 conf=0.92
  *  EM[fairness] value=+0.000 conf=1.00
  *  EM[fidelity] value=+0.600 conf=0.85
  *  EM[harm

## 2. Medical confidentiality vs duty to warn

In [4]:
ir_med = compile_document(
    EXAMPLES / "medical_confidentiality.txt",
    CompileOptions(tier=CompilerTier.RULES, extractor="mock"),
)
print(f"Verdict: {ir_med.deme_verdict.verdict}")
print(f"Canonical form: {ir_med.canonical_form}")
for c in ir_med.commitments:
    print(f"  - {c.id}: {c.type} ({c.status})")

Verdict: tragic_conflict_escalate
Canonical form: professional_privilege_versus_duty_to_warn
  - commitment_001: role_duty (active_but_defeasible)


## 3. Whistleblower

In [5]:
ir_whistle = compile_document(
    EXAMPLES / "whistleblower.txt",
    CompileOptions(tier=CompilerTier.RULES, extractor="mock"),
)
print(f"Verdict: {ir_whistle.deme_verdict.verdict}")
print(f"Canonical form: {ir_whistle.canonical_form}")

Verdict: tragic_conflict_escalate
Canonical form: institutional_loyalty_versus_public_truth_telling


## 4. The same nazi attic text with the rule extractor (no fixtures)

In [6]:
ir_nazi_rule = compile_document(
    EXAMPLES / "nazi_attic.txt",
    CompileOptions(tier=CompilerTier.RULES, extractor="rule"),
)
print(f"Rule-extractor verdict: {ir_nazi_rule.deme_verdict.verdict}")
print(f"Stakeholders detected: {[s.label for s in ir_nazi_rule.stakeholders]}")
print(f"Ethical facts detected: {[(f.kind, f.severity) for f in ir_nazi_rule.ethical_facts]}")

Rule-extractor verdict: tragic_conflict_escalate
Stakeholders detected: ['Document narrator/subject', 'village']
Ethical facts detected: [('coercion', 'grave'), ('legitimacy', 'grave'), ('externality', 'catastrophic'), ('deception', 'moderate')]


## 5. MoralVector timeline visualization

In [7]:
from erisml_compiler.viz.timeline_plot import save_timeline_plot
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

png_path = Path("out_quickstart") / "nazi_attic.png"
png_path.parent.mkdir(exist_ok=True)
save_timeline_plot(ir_nazi, png_path)

img = mpimg.imread(png_path)
plt.figure(figsize=(10, 5))
plt.imshow(img)
plt.axis('off')
plt.show()

C:\Users\abptl\AppData\Local\Temp\ipykernel_25168\652649611.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. EM-DAG outputs (per-module evaluations)

In [8]:
for name in sorted(ir_nazi.em_outputs):
    out = ir_nazi.em_outputs[name]
    deps = f"  (deps: {out.upstream_dependencies})" if out.upstream_dependencies else ""
    print(f"  {name:12} = {out.score.value:+.3f}  conf={out.score.confidence:.2f}{deps}")
    print(f"               {out.score.explanation}")

  autonomy     = -1.000  conf=0.95  (deps: ['legitimacy'])
               Autonomy assessment: No relevant facts detected. Composed with legitimacy assessment: consent under illegitimate authority is void.
  care         = +0.850  conf=0.93  (deps: ['harm'])
               Care assessment: Speaker is actively protecting vulnerable hidden refugees.
  epistemic    = -0.500  conf=0.90
               Epistemic assessment: Deception toward illegitimate murderous authority is permitted.
  externality  = -1.000  conf=0.92  (deps: ['harm'])
               Third-party externality: Catastrophic non-consensual risk imposed on village.
  fairness     = +0.000  conf=1.00
               Fairness assessment: No relevant facts detected.
  fidelity     = +0.600  conf=0.85  (deps: ['legitimacy'])
               1 active commitment(s); some defeasible.
  harm         = +0.000  conf=1.00
               Harm assessment: No relevant facts detected.
  legitimacy   = -1.000  conf=0.95
               Legitimac

## 7. Audit record

The audit record contains everything needed to reproduce or contest the verdict.

In [9]:
print(f"Compiler version : {ir_nazi.audit.compiler_version}")
print(f"Schema version   : {ir_nazi.audit.schema_version}")
print(f"Tier             : {ir_nazi.audit.tier}")
print(f"Extractor        : {ir_nazi.audit.extractor}")
print(f"EM-DAG profile   : {ir_nazi.audit.em_profile}")
print(f"Timestamp        : {ir_nazi.audit.timestamp_utc}")
print(f"IR hash          : {ir_nazi.audit.ir_hash}")
print(f"Source text hash : {ir_nazi.audit.source_text_hash}")
print()
print("Passes:")
for p in ir_nazi.audit.passes:
    print(f"  {p.pass_index:2d} {p.pass_name:35s} {p.duration_ms:.2f} ms")

Compiler version : 0.1.0
Schema version   : erisml_compiler_ir_v0.1
Tier             : rules
Extractor        : mock
EM-DAG profile   : default_em_dag_v0.1
Timestamp        : 2026-06-11T22:56:39.814268+00:00
IR hash          : 72d052bcc0c4b00ab6f398faa9d1bef2e1072aa4a7289f0afbca7e4f074311f6
Source text hash : 3b4c4c53f78260c99ac9248657ed26ef7b3de2a0385e90bbf166ab7aa35984f3

Passes:
   0 ingestion_text                      0.98 ms
   1 segmentation                        0.14 ms
   2 extraction_passes_2_through_7       0.11 ms
   8 tensorisation                       0.51 ms
  10 deme_evaluation                     0.02 ms
  11 conflict_detection                  0.05 ms
